# 문서 로더(Document Loader)

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

### Document 객체
: LangChain에서 Document 객체는 모든 데이터의 기본 단위이며, <br>
  Loader로 불러온 텍스트를 모델이 이해할 수 있는 표준 구조로 정리한 클래스

In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content="이 문서는 LangChain의 Document 객체 설명입니다.",
    metadata={"source": "lecture_note", "page": 1}
)
print('page_content:',doc.page_content)
print('metadata:',doc.metadata)

page_content: 이 문서는 LangChain의 Document 객체 설명입니다.
metadata: {'source': 'lecture_note', 'page': 1}


In [3]:
docs = \
[
  Document(
    page_content="1장. LangChain 개요...",
    metadata={"source": "data/guide.pdf", "page": 1}
  ),
  Document(
    page_content="2장. Document Loader의 종류...",
    metadata={"source": "data/guide.pdf", "page": 2}
  ),
]
print(docs[0])
print(docs[1])

page_content='1장. LangChain 개요...' metadata={'source': 'data/guide.pdf', 'page': 1}
page_content='2장. Document Loader의 종류...' metadata={'source': 'data/guide.pdf', 'page': 2}


## 문서 로더(Document Loader)
: LangChain의 Document Loader는 외부 문서(파일, 웹, 데이터베이스 등)를 읽어서  <br>
LangChain이 처리할 수 있는 Document 객체(list[Document]) 형태로 변환하는 구성요소이다.

###  [1] TextLoader
: TextLoader 는 LangChain에서 가장 기본적인 문서 로더(Document Loader) 로, <br>
일반 텍스트 파일(.txt)을 읽어들여 Document 객체로 변환한다.

In [4]:
%%writefile sample.txt
이 문서는 LangChain TextLoader 예시입니다.
여러 줄의 텍스트를 포함합니다.
TextLoader 는 LangChain에서 가장 기본적인 Document Loader이다

Overwriting sample.txt


In [5]:
from langchain_community.document_loaders import TextLoader

# 텍스트 파일 로드
loader = TextLoader("sample.txt", encoding="utf-8")
docs = loader.load()

print(docs[0].page_content)  # 파일의 내용 출력
print(docs[0].metadata)      # 파일 경로 등의 부가정보


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30776\1205535556.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


이 문서는 LangChain TextLoader 예시입니다.
여러 줄의 텍스트를 포함합니다.
TextLoader 는 LangChain에서 가장 기본적인 Document Loader이다

{'source': 'sample.txt'}


In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1) 텍스트 로드
loader = TextLoader("sample.txt", encoding="utf-8")
docs = loader.load()

# 2) 청크 단위로 나누기
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
split_docs = splitter.split_documents(docs)

# 3) 임베딩 생성 및 벡터 저장
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)

print("총", len(split_docs), "개의 청크가 저장되었습니다.")
split_docs

총 4 개의 청크가 저장되었습니다.


[Document(metadata={'source': 'sample.txt'}, page_content='이 문서는 LangChain TextLoader 예시입니다.'),
 Document(metadata={'source': 'sample.txt'}, page_content='여러 줄의 텍스트를 포함합니다.'),
 Document(metadata={'source': 'sample.txt'}, page_content='TextLoader 는 LangChain에서 가장 기본적인 Document'),
 Document(metadata={'source': 'sample.txt'}, page_content='Document Loader이다')]

###  [2] DirectoryLoader
: DirectoryLoader는 폴더(디렉터리) 안의 여러 파일을 자동으로 탐색하여 <br>
각 파일을 개별 Document 객체로 읽어들이는 문서 로더(Document Loader) 이다. <br>
이 클래스는 실제 프로젝트에서 대규모 문서 일괄 처리 시 거의 항상 사용되는 핵심 도구이다.

In [7]:
! mkdir data

���� ���͸� �Ǵ� ���� data��(��) �̹� �ֽ��ϴ�.


In [8]:
import os

# 1. data 폴더 생성 (이미 존재하면 무시)
os.makedirs("data", exist_ok=True)

# 2. 샘플 텍스트 데이터
samples = {
    "sample1.txt": """LangChain은 LLM(대규모 언어모델)을 활용한 파이프라인 구축을 위한 오픈소스 프레임워크입니다.
Document Loader를 통해 다양한 외부 데이터를 로드하고,
Text Splitter로 문서를 나눈 후, Embedding 및 VectorStore를 이용해 RAG 시스템을 구성할 수 있습니다.""",

    "sample2.txt": """FAISS는 Facebook AI Research에서 개발한 벡터 검색 라이브러리입니다.
LangChain에서는 문서 임베딩을 벡터로 변환한 후,
FAISS를 이용해 빠른 유사도 검색을 수행합니다.""",

    "sample3.txt": """pgvector는 PostgreSQL 데이터베이스에서 벡터 데이터를 저장하고 검색할 수 있게 해주는 확장 모듈입니다.
LangChain은 pgvector를 통해 RAG 시스템을 SQL 기반 환경에서도 구현할 수 있습니다."""
}

# 3. 파일 생성 및 저장
for filename, content in samples.items():
    path = os.path.join("data", filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f" {filename} 파일 생성 완료")

print("\n📂 'data' 폴더에 3개의 샘플 텍스트 파일이 준비되었습니다.")


 sample1.txt 파일 생성 완료
 sample2.txt 파일 생성 완료
 sample3.txt 파일 생성 완료

📂 'data' 폴더에 3개의 샘플 텍스트 파일이 준비되었습니다.


In [10]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data/",
    glob="**/*.txt",
    loader_cls=lambda path: TextLoader(path, encoding="utf-8"),  # UTF-8로 저장된 샘플 파일 로드
    show_progress=True
)

docs = loader.load()
print(len(docs), "개의 문서를 불러왔습니다.\n")
for i,doc in enumerate(docs):
    print(f'문서 {i+1}:\n', doc)
    print('-'*140)

100%|██████████| 3/3 [00:00<00:00, 149.75it/s]

3 개의 문서를 불러왔습니다.

문서 1:
 page_content='LangChain은 LLM(대규모 언어모델)을 활용한 파이프라인 구축을 위한 오픈소스 프레임워크입니다.
Document Loader를 통해 다양한 외부 데이터를 로드하고,
Text Splitter로 문서를 나눈 후, Embedding 및 VectorStore를 이용해 RAG 시스템을 구성할 수 있습니다.' metadata={'source': 'data\\sample1.txt'}
--------------------------------------------------------------------------------------------------------------------------------------------
문서 2:
 page_content='FAISS는 Facebook AI Research에서 개발한 벡터 검색 라이브러리입니다.
LangChain에서는 문서 임베딩을 벡터로 변환한 후,
FAISS를 이용해 빠른 유사도 검색을 수행합니다.' metadata={'source': 'data\\sample2.txt'}
--------------------------------------------------------------------------------------------------------------------------------------------
문서 3:
 page_content='pgvector는 PostgreSQL 데이터베이스에서 벡터 데이터를 저장하고 검색할 수 있게 해주는 확장 모듈입니다.
LangChain은 pgvector를 통해 RAG 시스템을 SQL 기반 환경에서도 구현할 수 있습니다.' metadata={'source': 'data\\sample3.txt'}
---------------------------------------------------------------------------------------------------------

### [3] PDF Loader

####  (1) PyPDFLoader
: PyPDFLoader는 PDF 파일을 텍스트로 변환하여 Document 객체로 만드는 로더입니다.<br>
내부적으로 pypdf (이전 명칭 PyPDF2) 라이브러리를 사용하며,
PDF의 각 페이지를 개별 Document 객체로 반환합니다.

In [ ]:
# ! pip install pypdf

In [11]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지.pdf")
docs = loader.load()

print("총", len(docs), "페이지 로드됨")
print(docs[0].page_content[:500])

총 16 페이지 로드됨
강화학습을 이용한 하천 녹조 발생 저감 모형 연구   47한국빅데이터학회지
제10권 제1호, 2025, pp. 47-62 https://doi.org/10.36498/kbigdt.2025.10.1.47
유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지: 
사용자 생성 콘텐츠에서의 극단성과 허위성 분류 
Decoding Review Anomalies: Classifying Extremity and Falsity in 
User-Generated Content
가중정1⋅김엘레나2⋅최재원3†
순천향대학교 경영학과1, 순천향대학교 경영학과2, 순천향대학교 경영학과3
요   약
본 연구는 YouTube 플랫폼의 호텔 리뷰를 대상으로 머신러닝과 자연어처리(NLP) 기법을 활용해 극단적 
및 조작된 리뷰를 식별⋅필터링하고자 한다. 소셜 미디어는 소비자 구매 결정에 중요한 영향을 미치며, 
사용자 생성 리뷰는 마켓플레이스 신뢰도의 핵심 요소로 작용한다. 그러나 일부 판매자들의 평점 조작과 
조작된 


In [12]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "./",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)
docs = loader.load()

print("총", len(docs), "개의 문서(페이지) 로드 완료")

Advanced encoding /UniKS-UTF16-H not implemented yet
Advanced encoding /UniKS-UTF16-H not implemented yet
Advanced encoding /UniKS-UTF16-H not implemented yet
Advanced encoding /UniKS-UTF16-H not implemented yet


총 290 개의 문서(페이지) 로드 완료


In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1) PDF 로드
loader = PyPDFLoader("유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지.pdf")
docs = loader.load()

# 2) 문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
split_docs = splitter.split_documents(docs)

# 3) 임베딩 및 벡터 저장
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)

print("PDF 문서가 벡터 DB로 성공적으로 저장되었습니다.")

PDF 문서가 벡터 DB로 성공적으로 저장되었습니다.


#### (2) PyMuPDF

In [ ]:
# !pip install pymupdf

In [14]:
from langchain_community.document_loaders import PyMuPDFLoader

# PyMuPDF 로더 인스턴스 생성
loader = PyMuPDFLoader("유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지.pdf")

# 문서 로드
docs = loader.load()

print("총", len(docs), "페이지 로드됨")

# 문서의 내용 출력
print(docs[10].page_content[:500])

총 16 페이지 로드됨
유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지   57
추출된 토픽 주제의 해석력을 높이기 위해, 
본 연구는 동시출현 점수(co-occurrence score)를 
계산하여 각 토픽의 의미적 일관성을 평가하였
다. 이 점수는 특정 주제 내에서 키워드들이 동
일한 문서 내에 얼마나 자주 함께 등장하는지를 
수치화한 것으로, 주제어 간의 상호 연관성을 
측정하는 지표로 활용된다. 수치화한 것으로, 주
제어 간의 상호 연관성을 측정하는 지표로 활용
된다. 동시출현 점수가 높을수록 키워드들이 밀
접하게 연결되어 있는 일관된 주제를 형성하며, 
예를 들어 ‘호텔 시설’, ‘청결’, ‘침대’, ‘조식’ 등
이 함께 자주 등장하는 경우, 해당 리뷰는 특정 
호텔의 물리적 조건에 집중된 논의임을 시사한
다. 반면, 동시출현 점수가 낮은 토픽은 단어 간 
맥락적 연결성이 약하거나, 리뷰 작성자들이 다
양한 맥락에서 단어를 더 자유롭게 사용하는 경
향이 있음을 나타낸다. 이는 주제가 분산되어 
있


#### (3) PDFMinerLoader

In [ ]:
# !pip install pdfminer.six

In [15]:
from langchain_community.document_loaders import PDFMinerLoader

# PDFMiner 로더 인스턴스 생성
loader = PDFMinerLoader("유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지.pdf")

# 데이터 로드
docs = loader.load()
print("총", len(docs), "페이지 로드됨")

# 문서의 내용 출력
print(docs[0].page_content[:500])

총 1 페이지 로드됨
강화학습을 이용한 하천 녹조 발생 저감 모형 연구   47한국빅데이터학회지제10권 제1호, 2025, pp. 47-62https://doi.org/10.36498/kbigdt.2025.10.1.47유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지: 사용자 생성 콘텐츠에서의 극단성과 허위성 분류 Decoding Review Anomalies: Classifying Extremity and Falsity in User-Generated Content가중정1⋅김엘레나2⋅최재원3†순천향대학교 경영학과1, 순천향대학교 경영학과2, 순천향대학교 경영학과3요  약본 연구는 YouTube 플랫폼의 호텔 리뷰를 대상으로 머신러닝과 자연어처리(NLP) 기법을 활용해 극단적 및 조작된 리뷰를 식별⋅필터링하고자 한다. 소셜 미디어는 소비자 구매 결정에 중요한 영향을 미치며, 사용자 생성 리뷰는 마켓플레이스 신뢰도의 핵심 요소로 작용한다. 그러나 일부 판매자들의 평점 조작과 조작된 리뷰 확산은 플랫폼의 신뢰


#### (4) PDFPlumber

In [ ]:
# !pip install pdfplumber

In [16]:
from langchain_community.document_loaders import PDFPlumberLoader

# PDF 문서 로더 인스턴스 생성
loader = PDFPlumberLoader("유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지.pdf")

# 문서 로딩
docs = loader.load()
print("총", len(docs), "페이지 로드됨")

# 첫 번째 문서 데이터 접근
print(docs[10].page_content[:500])

총 16 페이지 로드됨
유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지 57
추출된 토픽 주제의 해석력을 높이기 위해, 동시출현 네트워크는 사용자 리뷰의 다차원
본 연구는 동시출현 점수(co-occurrence score)를 적인 토픽 구조를 보여준다. 다양한 색상의 토
계산하여 각 토픽의 의미적 일관성을 평가하였 픽 노드는 여행 및 숙박 경험, 지역 리뷰, 동영상
다. 이 점수는 특정 주제 내에서 키워드들이 동 콘텐츠 피드백 등과 같은 다양한 의미적 시나리
일한 문서 내에 얼마나 자주 함께 등장하는지를 오를 다루며, 고빈도 키워드는 여러 토픽을 동
수치화한 것으로, 주제어 간의 상호 연관성을 시 발생 관계를 통해 서로 연결하여 리뷰 콘텐
측정하는 지표로 활용된다. 수치화한 것으로, 주 츠의 다양성과 집중도를 보여준다. <그림 2>는
제어 간의 상호 연관성을 측정하는 지표로 활용 시각화 된 동시출현 네트워크를 나타낸다.
된다. 동시출현 점수가 높을수록 키워드들이 밀 Word Cloud 시각화는 사용 시나리


#### (5) PyPDFium2

In [ ]:
# pip install pypdfium2

In [17]:
from langchain_community.document_loaders import PyPDFium2Loader

# PyPDFium2 로더 인스턴스 생성
loader = PyPDFium2Loader("유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지.pdf")

# 데이터 로드
docs = loader.load()
print("총", len(docs), "페이지 로드됨")

# 문서의 내용 출력
print(docs[10].page_content[:500])

총 16 페이지 로드됨
유튜브 기반 사용자 콘텐츠에서의 리뷰 이상 탐지 57
추출된 토픽 주제의 해석력을 높이기 위해, 
본 연구는 동시출현 점수(co-occurrence score)를 
계산하여 각 토픽의 의미적 일관성을 평가하였
다. 이 점수는 특정 주제 내에서 키워드들이 동
일한 문서 내에 얼마나 자주 함께 등장하는지를 
수치화한 것으로, 주제어 간의 상호 연관성을 
측정하는 지표로 활용된다. 수치화한 것으로, 주
제어 간의 상호 연관성을 측정하는 지표로 활용
된다. 동시출현 점수가 높을수록 키워드들이 밀
접하게 연결되어 있는 일관된 주제를 형성하며, 
예를 들어 ‘호텔 시설’, ‘청결’, ‘침대’, ‘조식’ 등
이 함께 자주 등장하는 경우, 해당 리뷰는 특정 
호텔의 물리적 조건에 집중된 논의임을 시사한
다. 반면, 동시출현 점수가 낮은 토픽은 단어 간 
맥락적 연결성이 약하거나, 리뷰 작성자들이 다
양한 맥락에서 단어를 더 자유롭게 사용하는 경
향이 있음을 나타낸다. 이는 주제가 분산되어 
있거나


#### (6) UnstructuredPDFLoader (실습 생략)
: UnstructuredPDFLoader는 PDF 문서의 텍스트, 표, 이미지 등 다양한 요소를 구조적으로 추출하는 LangChain의 고급 문서 로더입니다.<br>
기본적으로 unstructured 라이브러리를 사용하며, PyPDFLoader보다 문단 단위로 더 세밀한 텍스트 추출이 가능합니다. 

In [ ]:
# !pip install unstructured unstructured_inference pi_heif pdf2image

In [ ]:
# Windows 환경에는 poppler가  먼저 설치되어 있어야함

# from langchain_community.document_loaders import UnstructuredPDFLoader

# # 전체 텍스트를 단일 문서 객체로 변환
# loader = UnstructuredPDFLoader(pdf_filepath)
# pages = loader.load()

# len(pages)
# Windows 환경에는 poppler가  먼저 설치되어 있어야함

### [4] CSVLoader 와 DataFrameLoader

In [18]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# CSV 로더 생성
loader = CSVLoader(file_path="WHO_first9cols.csv")

# 데이터 로드
docs = loader.load()

print(len(docs))
print(docs[0].metadata)

202
{'source': 'WHO_first9cols.csv', 'row': 0}


In [19]:
import pandas as pd

# CSV 파일 읽기
df = pd.read_csv("WHO_first9cols.csv")
df

,Country,CountryID,Continent,Adolescent fertility rate (%),Adult literacy rate (%),Gross national income per capita (PPP international $),Net primary school enrolment ratio female (%),Net primary school enrolment ratio male (%),Population (in thousands) total
0,Afghanistan,1,1,151.0,28.0,NaN,NaN,NaN,26088.0
1,Albania,2,2,27.0,98.7,6000.0,93.0,94.0,3172.0
2,Algeria,3,3,6.0,69.9,5940.0,94.0,96.0,33351.0
3,Andorra,4,2,NaN,NaN,NaN,83.0,83.0,74.0
4,Angola,5,3,146.0,67.4,3890.0,49.0,51.0,16557.0
...,...,...,...,...,...,...,...,...,...
197,Vietnam,198,6,25.0,90.3,2310.0,91.0,96.0,86206.0
198,West Bank and Gaza,199,1,NaN,NaN,NaN,NaN,NaN,NaN
199,Yemen,200,1,83.0,54.1,2090.0,65.0,85.0,21732.0
200,Zambia,201,3,161.0,68.0,1140.0,94.0,90.0,11696.0


In [20]:
from langchain_community.document_loaders import DataFrameLoader

# 데이터 프레임 로더 설정, 페이지 내용 컬럼 지정
loader = DataFrameLoader(df, page_content_column="Country")

# 문서 로드
docs = loader.load()

# 데이터 출력
print(docs[0].page_content)

# 메타데이터 출력
print(docs[0].metadata)

Afghanistan
{'CountryID': 1, 'Continent': 1, 'Adolescent fertility rate (%)': 151.0, 'Adult literacy rate (%)': 28.0, 'Gross national income per capita (PPP international $)': nan, 'Net primary school enrolment ratio female (%)': nan, 'Net primary school enrolment ratio male (%)': nan, 'Population (in thousands) total': 26088.0}


### [5] JSONLoader

In [21]:
%%writefile employee.json
[
    {
        "name": "홍길동",
        "age": 30,
        "address": "서울특별시 강남구",
        "phoneNumbers": [
            {
                "type": "mobile",
                "number": "010-1234-5678"
            },
            {
                "type": "home",
                "number": "02-987-6543"
            }
        ]
    },
    {
        "name": "김철수",
        "age": 25,
        "address": "부산광역시 해운대구",
        "phoneNumbers": [
            {
                "type": "mobile",
                "number": "010-1111-2222"
            }
        ]
    },
    {
        "name": "이영희",
        "age": 28,
        "address": "대구광역시 수성구",
        "phoneNumbers": [
            {
                "type": "mobile",
                "number": "010-3333-4444"
            },
            {
                "type": "work",
                "number": "053-555-6666"
            }
        ]
    }
]

Overwriting employee.json


In [ ]:
# !pip install jq

In [22]:
from langchain_community.document_loaders import JSONLoader

# JSONLoader 생성
loader = JSONLoader(
    file_path="employee.json",
    jq_schema=".[].phoneNumbers",
    text_content=False,
)

# 문서 로드
docs = loader.load()

# 결과 출력
docs


[Document(metadata={'source': 'C:\\MyClaudeLab\\02_RAG 파이프라인 설계와 구현\\employee.json', 'seq_num': 1}, page_content='[{"type": "mobile", "number": "010-1234-5678"}, {"type": "home", "number": "02-987-6543"}]'),
 Document(metadata={'source': 'C:\\MyClaudeLab\\02_RAG 파이프라인 설계와 구현\\employee.json', 'seq_num': 2}, page_content='[{"type": "mobile", "number": "010-1111-2222"}]'),
 Document(metadata={'source': 'C:\\MyClaudeLab\\02_RAG 파이프라인 설계와 구현\\employee.json', 'seq_num': 3}, page_content='[{"type": "mobile", "number": "010-3333-4444"}, {"type": "work", "number": "053-555-6666"}]')]

### [6] WebBaseLoader
: URL(웹 페이지 주소) 를 입력받아,
그 웹 페이지의 본문 텍스트를 추출하여 LangChain의 Document 객체로 변환해주는 로더입니다. <br>
실제 서비스형 RAG나 뉴스 크롤링 실습할 때 가장 자주 쓰는 웹 문서 로더입니다.

In [23]:
from langchain_community.document_loaders import WebBaseLoader

# 로더 생성
loader = WebBaseLoader("https://blog.langchain.dev/langgraph-v0-2/")

# 웹 페이지 로드
docs = loader.load()

# 결과 확인
print(len(docs), "개의 문서 로드됨")
print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


1 개의 문서 로드됨
[Document(metadata={'source': 'https://blog.langchain.dev/langgraph-v0-2/', 'title': 'LangGraph v0.2: Increased customization with new checkpointer libraries', 'description': 'LangGraph v0.2 includes new checkpointer libraries for increased customization —\xa0including a SQLite checkpointer for local workflows and an optimized Postgres checkpointer to take your app to production. Plus, learn about LangGraph Cloud in open beta.', 'language': 'en'}, page_content="\nLangGraph v0.2: Increased customization with new checkpointer libraries\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nAgent ImprovementEngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceAgent InfrastructureDeploymentShip and scale agents in productionSandboxesRun agent-generated code safelyNo-Code AgentsFleetAgents for the whole companyOpen Source Frameworksdee

In [24]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# 여러 개의 url 지정 가능
url1 = "https://blog.langchain.dev/customers-replit/"
url2 = "https://blog.langchain.dev/langgraph-v0-2/"

loader = WebBaseLoader(
    web_paths=(url1, url2),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("article-header", "article-content")
        )
    ),
)
docs = loader.load()

# 결과 확인
print(len(docs), "개의 문서 로드됨")
print(docs)

2 개의 문서 로드됨
[Document(metadata={'source': 'https://blog.langchain.dev/customers-replit/'}, page_content=''), Document(metadata={'source': 'https://blog.langchain.dev/langgraph-v0-2/'}, page_content='')]


In [25]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.korea.kr/news/policyNewsView.do?newsId=148922373"
loader = WebBaseLoader(url)
docs = loader.load()

print(docs[0].metadata)
print(docs[0].page_content[:700])


{'source': 'https://www.korea.kr/news/policyNewsView.do?newsId=148922373', 'title': '[카툰공감] 중소기업의 든든한 버팀목 ‘납품대금 연동제’를 소개합니다 - 정책뉴스 | 뉴스 | 대한민국 정책브리핑', 'description': '중소기업의 든든한 버팀목 납품대금 연동제를 소개합니다 - 정책브리핑 | 뉴스 | 정책뉴스', 'language': 'ko'}



[카툰공감] 중소기업의 든든한 버팀목 ‘납품대금 연동제’를 소개합니다 - 정책뉴스 | 뉴스 | 대한민국 정책브리핑
































































































본문 바로가기
메인메뉴 바로가기



이 누리집은 대한민국 공식 전자정부 누리집입니다.






대한민국 정책브리핑



뉴스



뉴스



															
														정책뉴스



															
														부처별 뉴스



															
														정책포커스



															
														국민이 말하는 정책



															
														오피니언



															
														키워드 뉴스



															
														멀티미디어 뉴스







브리핑룸



브리핑룸



															
														보도자료



															
														사실은 이렇습니다



															
														브리핑 자료



															
														연설문



In [26]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1) 웹 문서 로드
loader = WebBaseLoader("https://ko.wikipedia.org/wiki/인공지능")
docs = loader.load()

# 2) 문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

# 3) 임베딩 + 벡터 저장
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)

print("총", len(split_docs), "개의 청크가 벡터DB에 저장되었습니다.")


총 154 개의 청크가 벡터DB에 저장되었습니다.


In [27]:
print(docs[0].page_content)





인공지능 - 위키백과, 우리 모두의 백과사전



























본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















외관
















기부

계정 만들기

로그인








개인 도구






기부


계정 만들기


로그인





























목차
사이드바로 이동
숨기기




처음 위치





1
강인공지능과 약인공지능




강인공지능과 약인공지능 하위 문단 토글하기





1.1
약인공지능










2
역사




역사 하위 문단 토글하기





2.1
인공지능 이론의 발전








2.2
인공지능의 탄생(1943-1956)






2.2.1
인공두뇌학과 초기 신경 네트워크








2.2.2
튜링 테스트








2.2.3
게임 인공지능








2.2.4
상징 추론과 논리 이론








2.2.5
다트머스 컨퍼런스 1956년: AI의 탄생










2.3
황금기(1956~1974년)






2.3.1
작업들






2.3.1.1
탐색 추리








2.3.1.2
자연어 처리








2.3.1.3
마이크로월드










2.3.2
낙관론








2.3.3
자금










2.4
AI의 첫번째 암흑기(1974-1980)






2.4.1
문제








2.4.2
자금 지원의 중단








2.4.3
캠퍼스 전역의 비판들








2.4.4
퍼셉트론과 연결망의 어두운 시대








2.4.5
깔끔이 : 논리, 프롤로그와 전문가 시스템








2

### [7] HWP Loader

In [ ]:
# ! pip install olefile

In [28]:
%%writefile hwploader.py
from typing import Any, Dict, List, Optional, Iterator
import olefile
import zlib
import struct
import re
import unicodedata
# from langchain.schema import Document
# from langchain.document_loaders.base import BaseLoader
from langchain_core.documents import Document    # 수정됨
from langchain_community.document_loaders.base import BaseLoader

class HWPLoader(BaseLoader):
    """HWP 파일 읽기 클래스. HWP 파일의 내용을 읽습니다."""

    def __init__(self, file_path: str, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.file_path = file_path
        self.extra_info = {"source": file_path}
        self._initialize_constants()

    def _initialize_constants(self) -> None:
        """상수 초기화 메서드"""
        self.FILE_HEADER_SECTION = "FileHeader"
        self.HWP_SUMMARY_SECTION = "\x05HwpSummaryInformation"
        self.SECTION_NAME_LENGTH = len("Section")
        self.BODYTEXT_SECTION = "BodyText"
        self.HWP_TEXT_TAGS = [67]

    def lazy_load(self) -> Iterator[Document]:
        """HWP 파일에서 데이터를 로드하고 표를 추출합니다.

        Yields:
            Document: 추출된 문서
        """
        load_file = olefile.OleFileIO(self.file_path)
        file_dir = load_file.listdir()

        if not self._is_valid_hwp(file_dir):
            raise ValueError("유효하지 않은 HWP 파일입니다.")

        result_text = self._extract_text(load_file, file_dir)
        yield self._create_document(text=result_text, extra_info=self.extra_info)

    def _is_valid_hwp(self, dirs: List[List[str]]) -> bool:
        """HWP 파일의 유효성을 검사합니다."""
        return [self.FILE_HEADER_SECTION] in dirs and [self.HWP_SUMMARY_SECTION] in dirs

    def _get_body_sections(self, dirs: List[List[str]]) -> List[str]:
        """본문 섹션 목록을 반환합니다."""
        section_numbers = [
            int(d[1][self.SECTION_NAME_LENGTH :])
            for d in dirs
            if d[0] == self.BODYTEXT_SECTION
        ]
        return [
            f"{self.BODYTEXT_SECTION}/Section{num}" for num in sorted(section_numbers)
        ]

    def _create_document(
        self, text: str, extra_info: Optional[Dict] = None
    ) -> Document:
        """문서 객체를 생성합니다."""
        return Document(page_content=text, metadata=extra_info or {})

    def _extract_text(
        self, load_file: olefile.OleFileIO, file_dir: List[List[str]]
    ) -> str:
        """모든 섹션에서 텍스트를 추출합니다."""
        sections = self._get_body_sections(file_dir)
        return "\n".join(
            self._get_text_from_section(load_file, section) for section in sections
        )

    def _is_compressed(self, load_file: olefile.OleFileIO) -> bool:
        """파일이 압축되었는지 확인합니다."""
        with load_file.openstream(self.FILE_HEADER_SECTION) as header:
            header_data = header.read()
            return bool(header_data[36] & 1)

    def _get_text_from_section(self, load_file: olefile.OleFileIO, section: str) -> str:
        """특정 섹션에서 텍스트를 추출합니다."""
        with load_file.openstream(section) as bodytext:
            data = bodytext.read()

        unpacked_data = (
            zlib.decompress(data, -15) if self._is_compressed(load_file) else data
        )

        text = []
        i = 0
        while i < len(unpacked_data):
            header, rec_type, rec_len = self._parse_record_header(
                unpacked_data[i : i + 4]
            )
            if rec_type in self.HWP_TEXT_TAGS:
                rec_data = unpacked_data[i + 4 : i + 4 + rec_len]
                text.append(rec_data.decode("utf-16"))
            i += 4 + rec_len

        text = "\n".join(text)
        text = self.remove_chinese_characters(text)
        text = self.remove_control_characters(text)
        return text

    @staticmethod
    def remove_chinese_characters(s: str):
        """중국어 문자를 제거합니다."""
        return re.sub(r"[\u4e00-\u9fff]+", "", s)

    @staticmethod
    def remove_control_characters(s):
        """깨지는 문자 제거"""
        return "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")

    @staticmethod
    def _parse_record_header(header_bytes: bytes) -> tuple:
        """레코드 헤더를 파싱합니다."""
        header = struct.unpack_from("<I", header_bytes)[0]
        rec_type = header & 0x3FF
        rec_len = (header >> 20) & 0xFFF
        return header, rec_type, rec_len

Overwriting hwploader.py


In [29]:
from hwploader import HWPLoader

#  한글 문서 로드
loader = HWPLoader("example.hwp")
docs = loader.load()

print("✅ 로드 완료")
print(docs[0].page_content[:500])

✅ 로드 완료
AI Agent 기술 개요와 활용 전략Ⅰ. 서론최근 인공지능 기술의 발전은 단순한 대화형 챗봇을 넘어,스스로 사고하고 행동하는 AI 에이전트(AI Agent) 의 시대를 열고 있다.AI 에이전트는 대규모 언어모델(LLM: Large Language Model)을 기반으로,자율적으로 목표를 설정하고, 외부 도구나 데이터베이스와 상호작용하여사람처럼 문제를 해결하는 새로운 형태의 인공지능 시스템이다.이러한 기술은 단순한 질문 응답을 넘어업무 자동화, 정보 탐색, 문서 생성, 분석 보고서 작성 등다양한 분야에서 실질적인 업무 대체 및 보조 역할을 수행하고 있다.AI 에이전트는 더 이상 ‘대화형 모델’이 아닌‘작업 수행자(Worker)’로 진화하고 있는 것이다.Ⅱ. AI 에이전트의 개념과 구조AI 에이전트는 기본적으로 다섯 가지 핵심 구성요소로 이루어진다.목표(Goal)사용자가 제시한 목적 또는 시스템이 스스로 정의한 과업.지각(Perception)입력된 데이터나 외부 환경의 상태를 인식하고


### [8] Docx2txtLoader(Word 문서 Loader)

In [ ]:
# 설치
# !pip install docx2txt

In [30]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("example.docx")  # 문서 로더 초기화

docs = loader.load()  # 문서 로딩

print("✅ 로드 완료")
print(docs[0].page_content[:500])

✅ 로드 완료
AI Agent 기술 개요와 활용 전략



Ⅰ. 서론



최근 인공지능 기술의 발전은 단순한 대화형 챗봇을 넘어,

스스로 사고하고 행동하는 AI 에이전트(AI Agent) 의 시대를 열고 있다.

AI 에이전트는 대규모 언어모델(LLM: Large Language Model)을 기반으로,

자율적으로 목표를 설정하고, 외부 도구나 데이터베이스와 상호작용하여

사람처럼 문제를 해결하는 새로운 형태의 인공지능 시스템이다.

이러한 기술은 단순한 질문 응답을 넘어

업무 자동화, 정보 탐색, 문서 생성, 분석 보고서 작성 등

다양한 분야에서 실질적인 업무 대체 및 보조 역할을 수행하고 있다.

AI 에이전트는 더 이상 ‘대화형 모델’이 아닌

‘작업 수행자(Worker)’로 진화하고 있는 것이다.



Ⅱ. AI 에이전트의 개념과 구조



AI 에이전트는 기본적으로 다섯 가지 핵심 구성요소로 이루어진다.

목표(Goal)

사용자가 제시한 목적 또는 시스템이 스스로 정의한 과업.
